# Tugas Praktikum #3
## Pemrosesan Paralel

**Nama:** Afelia Nasyifa  
**NIM:** 2411513010  
**Kelas:** B

---
## 1. Analisis Speedup Skala

Pada percobaan ini dilakukan perbandingan waktu eksekusi pencarian jumlah bilangan prima secara sekuensial dan menggunakan multiprocessing. Jumlah worker yang diuji adalah 1, 2, 4, dan 8.

Speedup dihitung dengan membandingkan waktu eksekusi sekuensial dengan waktu eksekusi paralel.

In [2]:
%%writefile latihan_nomor1.py

import multiprocessing as mp
import time

def hitung_prima(n):
    jumlah = 0

    for angka in range(2, n):
        prima = True

        for pembagi in range(2, int(angka ** 0.5) + 1):
            if angka % pembagi == 0:
                prima = False
                break

        if prima:
            jumlah += 1

    return jumlah


if __name__ == "__main__":

    daftar_n = [80000, 80000, 80000, 80000]

    # Perhitungan sekuensial
    mulai = time.perf_counter()

    hasil_sequential = []

    for n in daftar_n:
        hasil_sequential.append(hitung_prima(n))

    waktu_sequential = time.perf_counter() - mulai

    print("=== HASIL SEKUENSIAL ===")
    print("Hasil :", hasil_sequential)
    print(f"Waktu : {waktu_sequential:.2f} detik")
    print()

    # Perhitungan multiprocessing
    daftar_worker = [1, 2, 4, 8]

    print("=== HASIL MULTIPROCESSING ===")

    for worker in daftar_worker:

        mulai = time.perf_counter()

        with mp.Pool(processes=worker) as pool:
            hasil = pool.map(hitung_prima, daftar_n)

        waktu = time.perf_counter() - mulai

        speedup = waktu_sequential / waktu

        print(f"Worker  : {worker}")
        print(f"Hasil   : {hasil}")
        print(f"Waktu   : {waktu:.2f} detik")
        print(f"Speedup : {speedup:.2f}x")
        print("-" * 30)

Writing latihan_nomor1.py


In [3]:
import subprocess
import sys

hasil = subprocess.run(
    [sys.executable, "latihan_nomor1.py"],
    capture_output=True,
    text=True
)

print(hasil.stdout)

if hasil.stderr:
    print(hasil.stderr)

=== HASIL SEKUENSIAL ===
Hasil : [7837, 7837, 7837, 7837]
Waktu : 0.87 detik

=== HASIL MULTIPROCESSING ===
Worker  : 1
Hasil   : [7837, 7837, 7837, 7837]
Waktu   : 1.06 detik
Speedup : 0.83x
------------------------------
Worker  : 2
Hasil   : [7837, 7837, 7837, 7837]
Waktu   : 0.63 detik
Speedup : 1.38x
------------------------------
Worker  : 4
Hasil   : [7837, 7837, 7837, 7837]
Waktu   : 0.52 detik
Speedup : 1.66x
------------------------------
Worker  : 8
Hasil   : [7837, 7837, 7837, 7837]
Waktu   : 1.05 detik
Speedup : 0.83x
------------------------------



### Analisis

Dari percobaan terlihat bahwa penambahan jumlah worker tidak otomatis menghasilkan speedup yang sama dengan jumlah worker. Secara teori, jika pekerjaan dapat dibagi secara sempurna, penggunaan 2, 4, dan 8 worker dapat mendekati speedup 2x, 4x, dan 8x.

Pada kondisi nyata terdapat overhead ketika proses dibuat dan ketika pekerjaan dibagi ke beberapa proses. Selain itu, jumlah core CPU yang tersedia juga membatasi seberapa jauh proses dapat berjalan secara paralel.

Jika jumlah worker melebihi kemampuan CPU, penambahan worker tidak lagi memberikan peningkatan yang sebanding. Karena itu, kurva speedup aktual biasanya berada di bawah kurva ideal.

## 2. Shared Memory vs Message Passing

Percobaan ini membandingkan dua cara komunikasi antarproses. Pada metode pertama, setiap proses mengirim hasil perhitungannya melalui `multiprocessing.Queue`. Pada metode kedua, semua proses langsung memperbarui nilai yang berada pada shared memory menggunakan `multiprocessing.Value`.

Pada shared memory digunakan `Lock` agar proses tidak mengubah nilai bersama pada waktu yang bersamaan.

In [4]:
%%writefile latihan_nomor2.py

import multiprocessing as mp
import random
import time

JUMLAH_PROSES = 4
JUMLAH_ANGKA = 1_000_000


def hitung_queue(queue):
    total = 0

    random.seed()

    for _ in range(JUMLAH_ANGKA):
        total += random.randint(1, 100)

    queue.put(total)


def hitung_shared(shared_total, lock):
    total = 0

    random.seed()

    for _ in range(JUMLAH_ANGKA):
        total += random.randint(1, 100)

    with lock:
        shared_total.value += total


if __name__ == "__main__":

    # =========================
    # QUEUE
    # =========================

    queue = mp.Queue()

    proses = []

    mulai = time.perf_counter()

    for _ in range(JUMLAH_PROSES):
        p = mp.Process(
            target=hitung_queue,
            args=(queue,)
        )
        proses.append(p)
        p.start()

    hasil = []

    for _ in range(JUMLAH_PROSES):
        hasil.append(queue.get())

    for p in proses:
        p.join()

    waktu_queue = time.perf_counter() - mulai

    total_queue = sum(hasil)

    print("=== QUEUE ===")
    print("Hasil setiap proses :", hasil)
    print("Total akhir         :", total_queue)
    print(f"Waktu               : {waktu_queue:.2f} detik")
    print()


    # =========================
    # VALUE + LOCK
    # =========================

    shared_total = mp.Value("q", 0)
    lock = mp.Lock()

    proses = []

    mulai = time.perf_counter()

    for _ in range(JUMLAH_PROSES):
        p = mp.Process(
            target=hitung_shared,
            args=(shared_total, lock)
        )
        proses.append(p)
        p.start()

    for p in proses:
        p.join()

    waktu_shared = time.perf_counter() - mulai

    print("=== VALUE + LOCK ===")
    print("Total akhir :", shared_total.value)
    print(f"Waktu       : {waktu_shared:.2f} detik")
    print()


    # =========================
    # PERBANDINGAN
    # =========================

    print("=== PERBANDINGAN ===")
    print(f"Queue        : {waktu_queue:.2f} detik")
    print(f"Value + Lock : {waktu_shared:.2f} detik")

Writing latihan_nomor2.py


In [5]:
import subprocess
import sys

hasil = subprocess.run(
    [sys.executable, "latihan_nomor2.py"],
    capture_output=True,
    text=True
)

print(hasil.stdout)

if hasil.stderr:
    print(hasil.stderr)

=== QUEUE ===
Hasil setiap proses : [50542040, 50519423, 50496633, 50507261]
Total akhir         : 202065357
Waktu               : 1.52 detik

=== VALUE + LOCK ===
Total akhir : 201935686
Waktu       : 1.53 detik

=== PERBANDINGAN ===
Queue        : 1.52 detik
Value + Lock : 1.53 detik



### Analisis

Pada metode Queue, setiap proses menyelesaikan perhitungannya terlebih dahulu kemudian mengirim hasilnya ke proses utama. Data yang dikirim antarproses perlu melalui mekanisme komunikasi antarproses.

Pada metode Value, hasil langsung ditambahkan ke shared memory. Namun, karena nilai tersebut dapat diakses oleh beberapa proses, diperlukan `Lock` agar perubahan nilai tidak saling bertabrakan.

Hasil waktu dapat berbeda pada setiap komputer. Queue memiliki mekanisme message passing yang lebih jelas untuk pertukaran hasil, sedangkan Value dapat digunakan ketika beberapa proses perlu mengakses nilai bersama. Untuk data yang besar, pemilihan metode perlu mempertimbangkan jumlah data, frekuensi komunikasi, dan biaya sinkronisasi.

## 3. Producer-Consumer Antar-Proses

Pada percobaan ini pola producer-consumer dari threading diubah menjadi multiprocessing. Producer dan consumer dijalankan sebagai proses yang berbeda, sedangkan komunikasi data menggunakan `multiprocessing.Queue`.

In [6]:
import multiprocessing as mp
import time
import random


JUMLAH_DATA = 15
JUMLAH_CONSUMER = 3


def producer(queue):
    for i in range(1, JUMLAH_DATA + 1):
        data = f"Data-{i}"

        queue.put(data)

        print(f"[Producer] memasukkan {data}")
        time.sleep(random.uniform(0.05, 0.15))

    # Sinyal berhenti untuk setiap consumer
    for _ in range(JUMLAH_CONSUMER):
        queue.put(None)

    print("[Producer] selesai")


def consumer(queue, nama):
    while True:
        data = queue.get()

        if data is None:
            print(f"[{nama}] selesai")
            break

        print(f"[{nama}] memproses {data}")

        time.sleep(random.uniform(0.1, 0.25))


if __name__ == "__main__":
    queue = mp.Queue()

    producer_process = mp.Process(
        target=producer,
        args=(queue,)
    )

    consumer_processes = [
        mp.Process(
            target=consumer,
            args=(queue, f"Consumer-{i}")
        )
        for i in range(1, JUMLAH_CONSUMER + 1)
    ]

    for p in consumer_processes:
        p.start()

    producer_process.start()

    producer_process.join()

    for p in consumer_processes:
        p.join()

    print("\nSemua proses producer dan consumer selesai.")


Semua proses producer dan consumer selesai.


### Analisis

Secara konsep, pola producer-consumer pada multiprocessing masih sama dengan versi threading. Producer memasukkan data ke dalam queue dan consumer mengambil data tersebut untuk diproses.

Perbedaannya terdapat pada unit eksekusi dan media komunikasi. Pada threading, thread berada dalam satu proses sehingga menggunakan `queue.Queue`. Pada multiprocessing, producer dan consumer merupakan proses yang memiliki ruang memori sendiri sehingga digunakan `multiprocessing.Queue` untuk mengirim data antarproses.

Untuk pekerjaan CPU-bound, multiprocessing lebih sesuai dibanding threading karena setiap proses memiliki interpreter dan GIL sendiri. Dengan begitu beberapa proses dapat menggunakan core CPU yang berbeda.

## 4. Refleksi Singkat

### 1. Kapan saya memilih multiprocessing dibanding threading?

Saya akan memilih multiprocessing ketika pekerjaan yang dilakukan lebih banyak menggunakan CPU, misalnya perhitungan numerik, pengolahan data, atau proses yang membutuhkan komputasi cukup berat. Alasannya karena setiap proses memiliki interpreter dan GIL sendiri sehingga beberapa proses dapat berjalan pada core CPU yang berbeda.

Untuk pekerjaan yang lebih banyak menunggu I/O seperti mengambil data dari jaringan atau membaca banyak file, threading masih lebih praktis karena overhead-nya lebih kecil.

### 2. Kapan menggunakan Queue dibanding Value / Array?

Queue lebih cocok ketika data perlu dikirim dari satu proses ke proses lain dan proses tidak harus mengakses data yang sama secara langsung. Polanya lebih mudah dipahami karena satu proses memasukkan data dan proses lain mengambilnya.

Value atau Array lebih cocok ketika beberapa proses memang perlu menggunakan data yang berada pada shared memory. Namun, akses terhadap data tersebut tetap perlu diperhatikan karena beberapa proses dapat mengubah nilai pada waktu yang berdekatan. Untuk data yang dimodifikasi bersama, Lock diperlukan agar tidak terjadi race condition.

### 3. Kapan Manager lebih cocok?

Manager lebih cocok ketika data yang dibagikan memiliki struktur yang lebih kompleks, misalnya dictionary atau list yang digunakan oleh beberapa proses. Penggunaannya lebih sederhana dibanding mengatur shared memory secara langsung.

Kekurangannya adalah Manager memiliki overhead karena akses terhadap objek dilakukan melalui proses server Manager. Karena itu, untuk kebutuhan yang membutuhkan performa tinggi dan data sederhana, Value atau Array dapat menjadi pilihan yang lebih efisien.

### 4. Kesimpulan

Pemilihan multiprocessing, Queue, Value, Array, atau Manager tidak hanya ditentukan oleh jumlah proses yang digunakan. Jenis pekerjaan dan cara proses bertukar data juga perlu diperhatikan. Multiprocessing lebih cocok untuk pekerjaan CPU-bound, Queue digunakan untuk message passing, Value dan Array untuk shared memory sederhana, sedangkan Manager lebih praktis ketika struktur data yang digunakan lebih kompleks.